# CRISP-DM Notebook: Predicting `orders.is_fraud`

This notebook builds a full machine learning pipeline to predict fraud from a Postgres operational database, following CRISP-DM and Chapters 1-17 course skills.

## Scope
- Target: `orders.is_fraud`
- Source: Postgres (`DATABASE_URL`)
- Modeling goal: compare multiple classifiers and select a deployable fraud model
- Decision metrics: `F1`, `PR-AUC`, and `Recall` (with threshold analysis)

## Chapter Coverage Map
- **Ch 1**: Project framing via CRISP-DM
- **Ch 2-4**: Data wrangling and cleaning
- **Ch 6**: Feature-level exploration
- **Ch 7**: Automated preparation pipelines
- **Ch 8**: Relationship discovery
- **Ch 13**: Classification modeling
- **Ch 14**: Ensemble methods
- **Ch 15**: Evaluation, selection, and tuning
- **Ch 16**: Feature selection
- **Ch 17**: Deployment artifact serialization

In [1]:
import os
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import SelectFromModel

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")

ROOT = Path.cwd()
DATABASE_URL = os.getenv("DATABASE_URL", "").strip()
if not DATABASE_URL.startswith("postgresql://"):
    raise ValueError("Set DATABASE_URL to a PostgreSQL connection string (postgresql://...).")

ENGINE = create_engine(DATABASE_URL)
ARTIFACT_DIR = ROOT / "ml" / "models"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Using Postgres DATABASE_URL for source tables")

ValueError: Set DATABASE_URL to a PostgreSQL connection string (postgresql://...).

## 1) Business Understanding

Fraudulent transactions cause financial loss and operational overhead. The business objective is to flag suspicious orders early so risk teams can review high-risk transactions.

### Success Criteria
- Build a reproducible fraud model pipeline.
- Compare multiple models and select one based on **F1**, **PR-AUC**, and **Recall**.
- Produce deployable artifacts (serialized model + preprocessing + selected threshold).

### Modeling Decision Rule
Because fraud is typically imbalanced, we will prioritize:
1. Strong PR-AUC (ranking quality on positives)
2. Sufficient recall (catch fraudulent orders)
3. Good F1 balance for precision/recall tradeoff

In [ ]:
# Load operational tables from Postgres
customers = pd.read_sql_query("SELECT * FROM customers", ENGINE)
orders = pd.read_sql_query("SELECT * FROM orders", ENGINE)
order_items = pd.read_sql_query("SELECT * FROM order_items", ENGINE)
shipments = pd.read_sql_query("SELECT * FROM shipments", ENGINE)

print("customers", customers.shape)
print("orders", orders.shape)
print("order_items", order_items.shape)
print("shipments", shipments.shape)

orders.head(3)

## 2) Data Understanding (Ch. 6 + Ch. 8)

We first inspect quality and target balance, then explore feature distributions and fraud relationships.

In [ ]:
# Basic quality checks on orders (target table)
quality = pd.DataFrame({
    "dtype": orders.dtypes.astype(str),
    "null_count": orders.isna().sum(),
    "null_pct": (orders.isna().mean() * 100).round(2),
    "n_unique": orders.nunique(dropna=False)
}).sort_values("null_pct", ascending=False)

print("Target distribution (is_fraud):")
print(orders["is_fraud"].value_counts(dropna=False))
print("Fraud rate:", round(orders["is_fraud"].fillna(0).mean(), 4))
quality.head(20)

In [ ]:
# Feature engineering dataset (adapted from project scripts)
def build_fraud_dataset(orders, customers, shipments, order_items):
    orders = orders.copy()
    customers = customers.copy()
    shipments = shipments.copy()
    order_items = order_items.copy()

    item_aggs = (
        order_items.groupby("order_id")
        .agg(
            total_items=("quantity", "sum"),
            unique_products=("product_id", "nunique"),
            avg_item_price=("unit_price", "mean"),
        )
        .reset_index()
    )

    df = orders.merge(customers, on="customer_id", how="left", suffixes=("", "_cust"))
    df = df.merge(
        shipments[["order_id", "shipping_method", "distance_band", "promised_days", "actual_days"]],
        on="order_id",
        how="left",
    )
    df = df.merge(item_aggs, on="order_id", how="left")

    for c, fill in {
        "promo_code": "NONE",
        "city": "UNKNOWN",
        "state": "UNKNOWN",
        "zip_code": "00000",
        "shipping_zip": "00000",
        "billing_zip": "00000",
    }.items():
        if c in df.columns:
            df[c] = df[c].fillna(fill)

    for dt_col in ["order_datetime", "created_at", "birthdate"]:
        if dt_col in df.columns:
            df[dt_col] = pd.to_datetime(df[dt_col], errors="coerce")

    df["customer_age"] = ((df["order_datetime"] - df["birthdate"]).dt.days / 365.25).clip(14, 100)
    df["account_age_days"] = (df["order_datetime"] - df["created_at"]).dt.days.clip(lower=0)
    df["items_per_order"] = df["total_items"].fillna(0)
    df["avg_item_price"] = df["avg_item_price"].fillna(0)

    df["shipping_to_subtotal_ratio"] = np.where(
        df["order_subtotal"] > 0,
        df["shipping_fee"] / df["order_subtotal"],
        0,
    )
    df["tax_to_subtotal_ratio"] = np.where(
        df["order_subtotal"] > 0,
        df["tax_amount"] / df["order_subtotal"],
        0,
    )

    df["promo_used"] = df["promo_used"].fillna(0).astype(int)
    df["is_fraud"] = df["is_fraud"].fillna(0).astype(int)

    selected_columns = [
        "order_id", "customer_id", "order_total", "order_subtotal", "shipping_fee", "tax_amount",
        "risk_score", "payment_method", "device_type", "ip_country", "promo_used", "customer_segment",
        "loyalty_tier", "gender", "state", "shipping_state", "shipping_method", "distance_band",
        "promised_days", "actual_days", "items_per_order", "unique_products", "avg_item_price",
        "customer_age", "account_age_days", "shipping_to_subtotal_ratio", "tax_to_subtotal_ratio", "is_fraud"
    ]

    df = df[selected_columns].copy()

    for col in ["unique_products", "promised_days", "actual_days"]:
        df[col] = df[col].fillna(0)

    for col in [
        "payment_method", "device_type", "ip_country", "customer_segment", "loyalty_tier",
        "gender", "state", "shipping_state", "shipping_method", "distance_band"
    ]:
        df[col] = df[col].fillna("UNKNOWN")

    return df

fraud_df = build_fraud_dataset(orders, customers, shipments, order_items)
print("Fraud modeling dataset:", fraud_df.shape)
fraud_df.head(3)

In [ ]:
# Automated feature-level exploration (Ch 6)
num_cols = fraud_df.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in fraud_df.columns if c not in num_cols]

eda_summary = pd.DataFrame({
    "dtype": fraud_df.dtypes.astype(str),
    "missing_pct": (fraud_df.isna().mean() * 100).round(2),
    "unique": fraud_df.nunique(dropna=False),
})

print("Numeric columns:", len(num_cols), "| Categorical columns:", len(cat_cols))
eda_summary.sort_values(["missing_pct", "unique"], ascending=[False, False]).head(30)

In [ ]:
# Relationship discovery (Ch 8): fraud rate by key categorical features
for col in ["payment_method", "device_type", "ip_country", "customer_segment", "shipping_method", "distance_band"]:
    rel = (
        fraud_df.groupby(col, dropna=False)["is_fraud"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "fraud_rate"})
        .sort_values("fraud_rate", ascending=False)
    )
    print(f"\n{col} -> top fraud rates")
    display(rel.head(10))

In [ ]:
# Quick visual checks: class imbalance and top numeric relationships
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

fraud_df["is_fraud"].value_counts().sort_index().plot(kind="bar", ax=axes[0], title="Target Count (is_fraud)")
axes[0].set_xlabel("is_fraud")

sns.boxplot(data=fraud_df, x="is_fraud", y="risk_score", ax=axes[1])
axes[1].set_title("Risk Score by Fraud Label")

sns.boxplot(data=fraud_df, x="is_fraud", y="order_total", ax=axes[2])
axes[2].set_title("Order Total by Fraud Label")

plt.tight_layout()
plt.show()

## 3) Data Preparation + Modeling

We create an automated preprocessing pipeline and train multiple classification models:
- Logistic Regression (baseline classifier)
- Random Forest (ensemble, Ch 14)
- Gradient Boosting (ensemble, Ch 14)

Then we tune selected models and compare by F1, PR-AUC, and Recall.

In [ ]:
# Train/test split + preprocessing
id_cols = ["order_id", "customer_id"]
target_col = "is_fraud"

X = fraud_df.drop(columns=id_cols + [target_col], errors="ignore")
y = fraud_df[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_features = [c for c in X_train.columns if c not in num_features]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_features),
    ]
)

print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
print("Positive class rate (train):", round(y_train.mean(), 4))

In [ ]:
def score_model(name, model, X_train, y_train, X_test, y_test, threshold=0.5):
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= threshold).astype(int)
    return {
        "model": name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probs),
        "pr_auc": average_precision_score(y_test, probs),
        "cm": confusion_matrix(y_test, preds),
        "probs": probs,
    }

models = {
    "logistic_regression": Pipeline([
        ("preprocess", preprocess),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
    ]),
    "random_forest": Pipeline([
        ("preprocess", preprocess),
        ("clf", RandomForestClassifier(
            n_estimators=400,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ))
    ]),
    "gradient_boosting": Pipeline([
        ("preprocess", preprocess),
        ("clf", GradientBoostingClassifier(random_state=42))
    ]),
}

baseline_results = []
raw_outputs = {}
for name, model in models.items():
    out = score_model(name, model, X_train, y_train, X_test, y_test, threshold=0.5)
    raw_outputs[name] = out
    baseline_results.append({k: v for k, v in out.items() if k not in ["cm", "probs"]})

baseline_df = pd.DataFrame(baseline_results).sort_values(["pr_auc", "f1", "recall"], ascending=False)
baseline_df

In [ ]:
# Hyperparameter tuning (Ch 15): Random Forest as strong fraud candidate
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_pipeline = Pipeline([
    ("preprocess", preprocess),
    ("clf", RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
])

param_grid = {
    "clf__n_estimators": [200, 400],
    "clf__max_depth": [None, 10, 20],
    "clf__min_samples_leaf": [1, 2, 5],
}

grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="average_precision",  # PR-AUC proxy for imbalanced fraud
    cv=cv,
    n_jobs=-1,
    verbose=0,
)

grid.fit(X_train, y_train)

best_rf = grid.best_estimator_
rf_probs = best_rf.predict_proba(X_test)[:, 1]

print("Best params:", grid.best_params_)
print("Best CV PR-AUC:", round(grid.best_score_, 4))

In [ ]:
# Threshold analysis: optimize for F1 while inspecting PR-AUC/Recall trade-off
threshold_grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
rows = []
for t in threshold_grid:
    preds = (rf_probs >= t).astype(int)
    rows.append({
        "threshold": t,
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
    })

thr_df = pd.DataFrame(rows)
best_thr_row = thr_df.sort_values(["f1", "recall"], ascending=False).iloc[0]
selected_threshold = float(best_thr_row["threshold"])

final_preds = (rf_probs >= selected_threshold).astype(int)
final_metrics = {
    "threshold": selected_threshold,
    "accuracy": accuracy_score(y_test, final_preds),
    "precision": precision_score(y_test, final_preds, zero_division=0),
    "recall": recall_score(y_test, final_preds, zero_division=0),
    "f1": f1_score(y_test, final_preds, zero_division=0),
    "roc_auc": roc_auc_score(y_test, rf_probs),
    "pr_auc": average_precision_score(y_test, rf_probs),
}

print("Selected threshold:", selected_threshold)
pd.DataFrame([final_metrics]).T.rename(columns={0: "value"})

In [ ]:
# Feature selection (Ch 16): SelectFromModel + compare performance
feature_select_pipeline = Pipeline([
    ("preprocess", preprocess),
    (
        "selector",
        SelectFromModel(
            estimator=RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            ),
            threshold="median",
        ),
    ),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
])

fs_out = score_model(
    "feature_select_logistic",
    feature_select_pipeline,
    X_train,
    y_train,
    X_test,
    y_test,
    threshold=0.5,
)

pd.DataFrame([
    {k: v for k, v in fs_out.items() if k not in ["cm", "probs"]}
]).T.rename(columns={0: "value"})

In [ ]:
# Consolidated model comparison table
comparison_rows = baseline_df[["model", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]].copy()
comparison_rows = pd.concat([
    comparison_rows,
    pd.DataFrame([
        {
            "model": "random_forest_tuned_thresholded",
            "accuracy": final_metrics["accuracy"],
            "precision": final_metrics["precision"],
            "recall": final_metrics["recall"],
            "f1": final_metrics["f1"],
            "roc_auc": final_metrics["roc_auc"],
            "pr_auc": final_metrics["pr_auc"],
        },
        {
            "model": "feature_select_logistic",
            "accuracy": fs_out["accuracy"],
            "precision": fs_out["precision"],
            "recall": fs_out["recall"],
            "f1": fs_out["f1"],
            "roc_auc": fs_out["roc_auc"],
            "pr_auc": fs_out["pr_auc"],
        },
    ])
], ignore_index=True)

comparison_rows.sort_values(["pr_auc", "f1", "recall"], ascending=False)

## 4) Evaluation + Selection Decision

Decision approach used:
- Compare all candidate models by **PR-AUC**, **F1**, and **Recall**.
- Use threshold tuning on the top imbalanced-learning candidate.
- Prefer the model-threshold pair with strongest fraud capture quality and stable precision.

Use the table above as your submission evidence, then keep that winning model for deployment.

In [ ]:
# Deployment artifacts (Ch 17): serialize model package + metadata
final_model = best_rf
artifact = {
    "model": final_model,
    "selected_threshold": selected_threshold,
    "feature_columns": X_train.columns.tolist(),
    "target": target_col,
    "metrics": final_metrics,
}

artifact_path = ARTIFACT_DIR / "fraud_pipeline_artifact.joblib"
joblib.dump(artifact, artifact_path)

print("Saved artifact:", artifact_path)
print("Selected threshold:", selected_threshold)
print("Final metrics:")
pd.DataFrame([final_metrics])

In [ ]:
# Inference demo: score one sample record
loaded = joblib.load(artifact_path)
model = loaded["model"]
thr = loaded["selected_threshold"]

sample = X_test.iloc[[0]].copy()
prob = float(model.predict_proba(sample)[:, 1][0])
pred = int(prob >= thr)

print("Sample fraud probability:", round(prob, 4))
print("Threshold:", thr)
print("Predicted class:", pred)
sample

## 5) Final Conclusion (for report/submission)

- We implemented a full CRISP-DM fraud pipeline from business framing to deployment artifacts.
- We compared multiple models and selected based on F1, PR-AUC, and Recall.
- We applied model tuning and feature selection and serialized the final artifact for integration into the deployed app pipeline.

### Suggested next improvements
- Add probability calibration and cost-sensitive threshold selection based on business cost of false negatives.
- Add temporal validation split to mimic real production drift.
- Add model monitoring metrics post-deployment.